# LayerNorm

源码导航：[core/norm/layer_norm.py](../../../core/norm/layer_norm.py) 中的 `LayerNorm`。

Ba et al. (2016) 在 *Layer Normalization* 中提出 **LayerNorm**，通过对单个样本的所有特征维度进行均值-方差白化，解决深度神经网络中内部协变量偏移（Internal Covariate Shift）问题。与 BatchNorm 不同，LayerNorm 不依赖批次统计量，因此在序列建模（NLP、语音识别等变长输入场景）中具有天然优势。GPT-2 及早期 Transformer 均采用 LayerNorm 作为默认归一化方案。

### 1. 理论推导

设 $x \in \mathbb{R}^d$ 为某一 token 的隐状态向量，LayerNorm 的计算分为两步：

**（1）中心化**

减去均值以消除特征间的共同偏移：

$$\hat{x}_i = x_i - \mu, \quad \mu = \frac{1}{d}\sum_{j=1}^{d} x_j$$

**（2）缩放**

除以标准差，使输出具有单位方差：

$$\text{LayerNorm}(x)_i = \frac{\hat{x}_i}{\sigma + \epsilon} \cdot w_i + b_i$$

其中：
- $\sigma = \sqrt{\frac{1}{d}\sum_{j=1}^{d} \hat{x}_j^2 + \epsilon}$ 为标准差；
- $\epsilon$ 为防止除零的数值稳定项（默认 $10^{-5}$）；
- $w \in \mathbb{R}^d$ 为可学习缩放参数，初始化为全 $\mathbf{1}$；
- $b \in \mathbb{R}^d$ 为可学习偏移参数，初始化为全 $\mathbf{0}$（可选）。

**与 RMSNorm 的对比：**

| 属性 | LayerNorm | RMSNorm |
|---|---|---|
| 中心化（减均值） | **✓** | ✗ |
| 可学习 scale $w$ | ✓ | ✓ |
| 可学习 bias $b$ | ✓（可选）| ✗ |
| 归一化统计量 | 标准差 $\sigma$ | 均方根 RMS |
| 每维参数量（dim=$d$）| $2d$（含 bias）| $d$ |

后续研究表明，在 decoder-only 的大语言模型中，**中心化操作对训练稳定性的贡献远小于尺度归一化**，这也是 RMSNorm 成为现代 LLM（LLaMA、Qwen 等）首选的原因。但在 encoder-only 或早期 Transformer 中，LayerNorm 仍是最稳健的基线。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.layer_norm import LayerNorm
from core.norm.rmsnorm import RMSNorm

### 2. 形状与数值检查

In [7]:
torch.manual_seed(0)
# 构造均值不为零、方差约为 3 的随机激活
x = torch.randn(2, 4, 16) * 3 + 2.0
norm = LayerNorm(16, eps=1e-5)
y = norm(x)

print("x.shape =", tuple(x.shape))
print("y.shape =", tuple(y.shape))
print("均值 before:", x.mean(dim=-1).mean().item())
print("均值 after :", y.mean(dim=-1).mean().item())
print("标准差 before:", x.std(dim=-1).mean().item())
print("标准差 after :", y.std(dim=-1).mean().item())
assert x.shape == y.shape, "LayerNorm 必须保持输入输出维度一致！"
# 验证归一化后每 token 的均值接近 0，标准差接近 1
assert y.mean(dim=-1).abs().max().item() < 1e-5, "归一化后均值应接近 0！"
assert (y.std(dim=-1) - 1.0).abs().max().item() < 1e-1, "归一化后标准差应接近 1！"
print("\n✅ LayerNorm 数值验证通过")

x.shape = (2, 4, 16)
y.shape = (2, 4, 16)
均值 before: 2.1573235988616943
均值 after : -3.4924596548080444e-09
标准差 before: 3.175750732421875
标准差 after : 1.0327949523925781

✅ LayerNorm 数值验证通过


### 3. 与 RMSNorm 的参数量对比

In [8]:
dim = 1536
ln = LayerNorm(dim, bias=True)
rms = RMSNorm(dim)

ln_params = sum(p.numel() for p in ln.parameters())
rms_params = sum(p.numel() for p in rms.parameters())

print(f"LayerNorm params: {ln_params:,}  (scale w + bias b)")
print(f"RMSNorm   params: {rms_params:,}  (仅 scale w)")
print(f"RMSNorm 相比 LayerNorm 节省: {(1 - rms_params / ln_params) * 100:.0f}%")

LayerNorm params: 3,072  (scale w + bias b)
RMSNorm   params: 1,536  (仅 scale w)
RMSNorm 相比 LayerNorm 节省: 50%


### 4. Bias 参数的作用

LayerNorm 的 `bias` 参数允许输出分布整体平移。关闭 bias（`bias=False`）后，LayerNorm 退化为"中心化 + 缩放"，输出均值严格为 0；开启 bias 后，均值由 bias 的初始值（全 0）决定，但在训练过程中 bias 会被更新，使模型能够学习最优的输出偏移。

In [ ]:
x = torch.randn(1, 1, 8) + 5.0

ln_with_bias = LayerNorm(8, bias=True)
ln_no_bias = LayerNorm(8, bias=False)

y_bias = ln_with_bias(x)
y_no_bias = ln_no_bias(x)

print("输入均值:", x.mean(dim=-1).item())
print("带 bias 输出均值:", y_bias.mean(dim=-1).item())
print("无 bias 输出均值:", y_no_bias.mean(dim=-1).item())
print("带 bias 输出标准差:", y_bias.std(dim=-1).item())
print("无 bias 输出标准差:", y_no_bias.std(dim=-1).item())
assert (y_no_bias.mean(dim=-1).abs() < 1e-5).item(), "无 bias 时均值应为 0"

### 5. 源码精讲

以下为 `core/norm/layer_norm.py` 的完整实现：

```python
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5, bias: bool = True) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape)) if bias else None
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.nn.functional.layer_norm(
            x, self.weight.shape, self.weight, self.bias, self.eps
        )
```

关键设计点：
- 直接复用 PyTorch 原生的 `F.layer_norm`，保证数值稳定性与计算效率。
- `bias` 为可选参数，关闭时可节省 $d$ 个参数，便于与 RMSNorm 做公平对比。
- 不对输入做 fp32 上溯，因为 `F.layer_norm` 内部已实现混合精度安全逻辑。
- 与项目其他 norm 模块保持统一接口：`normalized_shape: int`, `eps: float`, `bias: bool`。

---

## 延伸阅读与参考资料

### 核心论文
- **Layer Normalization**: Ba et al., 2016. [arXiv:1607.06450](https://arxiv.org/abs/1607.06450)

### 工程实践
- **GPT-2**: Radford et al., 2019. — 使用带 bias 的标准 LayerNorm
- **BERT**: Devlin et al., 2018. — Encoder-only 模型的 LayerNorm 应用
- **RMSNorm**: Zhang and Sennrich, 2019. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467) — LayerNorm 的简化后继